# **De Novo Protein-Binder Design Prototype**

In this notebook, we walk through an end-to-end protein binder design pipeline against the insulin receptor (`examples/input_pdbs/insulin_target.pdb`, chain A 1-150, hotspots A59/A83/A91).

This notebook is suitable for prototype runs. For large scale runs, the whole pipeline is managed by SnakeMake.

---

## **1. Setup and environment check**

All pipeline logic lives in `pipeline/`, so the production `Snakefile` can execute the same code paths. Parameters come from `config/campaign.yaml` —
change them there, not here.

**Two flags control what this notebook does.**:
* `DRY_RUN` prints the Docker commands without running them, which is how the notebook is validated on a machine with no GPU.
* `SMOKE_TEST` shrinks the pipeline and records metrics without enforcing gates.

**Read the smoke-test result as a plumbing check, not a scientific one.** At this scale the pipeline is expected to yield 0-1 leads. What matters is that every
metric column is populated: a NaN column means a parser needs fixing, and `Campaign.health()` reports exactly that.

In [ ]:
import os, sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# Add repo to system path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from pipeline import docker, hotspots, metrics, parsers, pdb_io, report, trb, viz

# Ignore warnings
warnings.simplefilter("ignore")

# Set limits to displayed Pandas dataframes
pd.set_option("display.width", 160, "display.max_columns", 50)

# Set repo path and change directpry ito it
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)

# Read binder design campaign config 
CONFIG = yaml.safe_load((REPO / "config/campaign.yaml").read_text())

# DRY_RUN: print container commands instead of executing them. Defaults to True
# when no GPU is visible, so the notebook is safe to open anywhere.
DRY_RUN = not docker.gpu_available()
SMOKE_TEST = CONFIG["campaign"]["smoke_test"]
SCALE = "smoke" if SMOKE_TEST else "full"

print(f"repo       : {REPO}")
print(f"GPU        : {'yes' if docker.gpu_available() else 'no'}")
print(f"DRY_RUN    : {DRY_RUN}")
print(f"SMOKE_TEST : {SMOKE_TEST}  (scale='{SCALE}')")
print(f"cache      : {docker.cache_dir()}")

In [ ]:
# Which images exist. Missing images are fine in DRY_RUN; they are blocking
# otherwise, so check before spending time on Section 2.
import subprocess

for name, spec in CONFIG["containers"].items():
    image = spec["image"]
    present = subprocess.run(
        ["docker", "image", "inspect", image],
        capture_output=True,
    ).returncode == 0
    print(f"  {'ok  ' if present else '-- '} {image}")

if not DRY_RUN:
    # RFdiffusion's CUDA 11.6 pin emits no sm_90 kernels, so an H100 host fails
    # at the first tensor copy, minutes into a run. Fail here instead.
    print()
    print(docker.check_gpu_compatibility(CONFIG["containers"]["rfdiffusion"]["image"]))

---

## **2. Hotspot mapping and target preparation**

The hotspots are the published set from `examples/design_ppi.sh`, not a geometric guess: `select_patch` proposes a valid but *different* patch on this target, because epitope choice carries biology the geometry does not. Geometry is used here only to confirm the published set is exposed and contiguous.

In [ ]:
target_cfg = CONFIG["target"]
target_pdb = REPO / target_cfg["pdb"]
chain_id = target_cfg["chain"]
HOTSPOTS = list(target_cfg["hotspots"])

struct = pdb_io.read_pdb(target_pdb)
print(pdb_io.describe(struct))

target_chain = struct[chain_id]
print(f"\nhotspots: {HOTSPOTS}")

In [ ]:
# Real SASA (Shrake-Rupley, via Biopython) rather than a burial proxy.
rel = hotspots.relative_sasa(target_pdb, chain_id)
absolute = hotspots.compute_sasa(target_pdb, chain_id)

rows = []
for label in HOTSPOTS:
    _, rid = pdb_io.parse_label(label)
    res = next(r for r in target_chain.residues if r.seq_id == rid)
    rows.append({
        "hotspot": label, "residue": res.name,
        "sasa_A2": round(absolute[rid], 1),
        "relative_sasa": round(rel.get(rid, float("nan")), 3),
    })
print(pd.DataFrame(rows).to_string(index=False))

median = np.median(list(absolute.values()))
print(f"\nmedian SASA across chain {chain_id}: {median:.1f} A^2")

In [ ]:
# Patch geometry. This is only a sanity check that the
# residues describe one site a single binder could engage.
d = hotspots.patch_distances(target_chain, HOTSPOTS)
print(pd.DataFrame(d, index=HOTSPOTS, columns=HOTSPOTS).round(2).to_string())

spread = d[~np.eye(len(HOTSPOTS), dtype=bool)].max()
print(f"\nmax pairwise Cb-Cb: {spread:.2f} A  (sanity limit {hotspots.MAX_PATCH_DISTANCE} A)")

In [ ]:
# Checkpoint 1: target surface grey, hotspots dark red sticks.
viz.write_pml_hotspots(target_pdb, HOTSPOTS, REPO / "results/pml/01_hotspots.pml",
                       chain=chain_id)
viz.show_target_hotspots(target_pdb, HOTSPOTS, chain=chain_id)

---

## **3. Backbone generation (RFdiffusion)**

Contig `[A1-150/0 70-100]`: the target chain, a chain break so the binder is not fused to it, then a 70-100 residue de novo binder.

After generation, every design is checked against its `.trb`: chain roles, input-to-design renumbering, and whether the binder actually engaged the hotspots
it was conditioned on.

In [ ]:
rf = CONFIG["rfdiffusion"]
n_designs = rf["num_designs"][SCALE]
backbone_dir = REPO / CONFIG["paths"]["backbones"]
backbone_dir.mkdir(parents=True, exist_ok=True)

spec = docker.ContainerRun(
    image=CONFIG["containers"]["rfdiffusion"]["image"],
    command=[
        f"inference.output_prefix=/work/{CONFIG['paths']['backbones']}/design",
        f"inference.input_pdb=/work/{target_cfg['pdb']}",
        f"contigmap.contigs={rf['contigs']}",
        f"ppi.hotspot_res=[{','.join(HOTSPOTS)}]",
        f"inference.num_designs={n_designs}",
        f"inference.ckpt_override_path=/cache/rfdiffusion/models/{rf['checkpoint']}",
        f"denoiser.noise_scale_ca={rf['noise_scale_ca']}",
        f"denoiser.noise_scale_frame={rf['noise_scale_frame']}",
        f"inference.write_trajectory={rf['write_trajectory'][SCALE]}",
    ],
    mounts=[docker.Mount(REPO, "/work")],
    workdir="/work",
)

print(f"generating {n_designs} backbone(s)\n")
docker.run(spec, dry_run=DRY_RUN)

In [ ]:
# Step 1 QC. Chain roles and residue numbering come from each .trb:
# contig order makes the TARGET chain A and the binder chain B.
designs = sorted(backbone_dir.glob("design_*.pdb"))
print(f"{len(designs)} design(s) found\n")

qc_rows = []
for pdb in designs:
    info = trb.read_trb(trb.trb_path_for(pdb))
    roles = trb.classify_chains(info, pdb)
    row = {"design": pdb.stem, "binder": roles.binder_chain,
           "target": roles.target_chain, "binder_len": roles.binder_idx0.size,
           "rfdiff_plddt": round(float(info.final_plddt().mean()), 1)}
    if roles.is_binder_design:
        check = hotspots.verify_design_hotspots(pdb)
        row.update({"hotspots_engaged": f"{check['n_engaged']}/{len(HOTSPOTS)}",
                    "min_cb": min(check["min_cb_distance"].values())})
    else:
        row["hotspots_engaged"] = f"n/a ({roles.detail[:40]})"
    qc_rows.append(row)

if qc_rows:
    print(pd.DataFrame(qc_rows).to_string(index=False))
    print("\nDesigns whose binder never reached a hotspot ignored their")
    print("conditioning and should not consume Step 2 compute.")

In [ ]:
# Checkpoint 2: binding-mode diversity across the ensemble.
if designs:
    viz.write_pml_superposition(designs[0], designs[0],
                                REPO / "results/pml/02_backbones.pml")
    display(viz.show_backbone_ensemble(designs, target_pdb=target_pdb,
                                       target_chain="A", binder_chain="B"))

---

## **4. Fixed-backbone sequence design (ProteinMPNN)**

SolubleMPNN weights at T=0.1, designing the binder chain only with target coordinates fixed. 

Scores come from ProteinMPNN's own FASTA headers. `score` is a mean negative log-likelihood, so **lower is better** — a filter written `score > x` selects the worst sequences.

In [ ]:
mpnn = CONFIG["proteinmpnn"]
n_seqs = mpnn["seqs_per_backbone"][SCALE]
seq_dir = REPO / CONFIG["paths"]["sequences"]
seq_dir.mkdir(parents=True, exist_ok=True)

# Design the diffused chain, fix the target. Roles are read from the .trb
# rather than taken from config.
if designs:
    roles = trb.classify_chains(trb.read_trb(trb.trb_path_for(designs[0])), designs[0])
    design_chain = roles.binder_chain or mpnn["design_chains"][0]
    fixed_chain = roles.target_chain or mpnn["fixed_chains"][0]
else:
    design_chain, fixed_chain = mpnn["design_chains"][0], mpnn["fixed_chains"][0]

spec = docker.ContainerRun(
    image=CONFIG["containers"]["proteinmpnn"]["image"],
    command=[
        "--pdb_path_chains", design_chain,
        "--out_folder", f"/work/{CONFIG['paths']['sequences']}",
        "--num_seq_per_target", str(n_seqs),
        "--sampling_temp", str(mpnn["sampling_temperature"]),
        "--batch_size", "1",
    ] + (["--use_soluble_model"] if mpnn["use_soluble_weights"] else []),
    mounts=[docker.Mount(REPO, "/work")],
    workdir="/work",
)
print(f"designing {n_seqs} seq(s) per backbone; design={design_chain} fixed={fixed_chain}\n")
docker.run(spec, dry_run=DRY_RUN)

In [ ]:
# Parse MPNN's own scores; write a binder-only FASTA for Step 3.
# The target chain MUST be dropped here -- folding the concatenated
# multi-chain string would make every downstream scRMSD meaningless.
all_samples = []
for fasta in sorted((seq_dir / "seqs").glob("*.fa")) or sorted(seq_dir.glob("*.fa")):
    native, samples = parsers.parse_proteinmpnn_fasta(fasta, design_id=fasta.stem)
    all_samples.extend(samples)

if all_samples:
    binder_fa = parsers.binder_fasta(all_samples, seq_dir / "binders.fasta")
    frame = pd.DataFrame([s.as_row() for s in all_samples])
    print(frame.sort_values("mpnn_score").to_string(index=False))
    print(f"\n{len(all_samples)} sequences -> {binder_fa}")
else:
    print("no sequences yet (expected in DRY_RUN)")

---

## **5. Monomer self-consistency (Step 3)**

Two tiers: ESMFold as a cheap high-throughput filter, then Boltz-2 for all-atom confirmation.

Thresholds are the relaxed, literature-aligned values in `config/campaign.yaml` (3A: pLDDT ≥ 80, scRMSD ≤ 2.0 Å; 3B: pLDDT ≥ 85, scRMSD ≤ 1.5 Å).

In [ ]:
campaign = report.Campaign()
if all_samples:
    campaign.add_all([s.as_row() for s in all_samples])

esm_dir = REPO / CONFIG["paths"]["esmfold"]
spec = docker.ContainerRun(
    image=CONFIG["containers"]["esmfold"]["image"],
    command=["--fasta", f"/work/{CONFIG['paths']['sequences']}/binders.fasta",
             "--out-dir", f"/work/{CONFIG['paths']['esmfold']}"],
    mounts=[docker.Mount(REPO, "/work")],
    workdir="/work",
)
docker.run(spec, dry_run=DRY_RUN)

In [ ]:
# Gate 3A. enforce=False under SMOKE_TEST: metrics are recorded and the verdict
# computed, but nothing is dropped, so a parser bug is visible as MISSING rather
# than emptying the funnel.
records = []
for sample in all_samples:
    design_pdb = backbone_dir / f"{sample.design_id}.pdb"
    try:
        records.append(parsers.parse_esmfold(
            esm_dir, sample.design_id, sample.sequence_id,
            design_pdb=design_pdb if design_pdb.exists() else None,
            binder_chain=design_chain,
        ))
    except parsers.ParseError as exc:
        print(f"  parse failed for {sample.sequence_id}: {exc}")

campaign.add_records(records)
survivors = campaign.gate("gate3a", CONFIG["gates"]["step3a_esmfold"],
                          "esmfold", enforce=not SMOKE_TEST)
print(f"\ngate 3A: {len(survivors)} of {len(records)} passed")
if not campaign.funnel().empty:
    print(campaign.funnel().to_string(index=False))

In [ ]:
# Gate 3B: Boltz-2 all-atom monomer confirmation on the 3A survivors.
boltz_dir = REPO / CONFIG["paths"]["boltz2_monomer"]
spec = docker.ContainerRun(
    image=CONFIG["containers"]["boltz2"]["image"],
    command=[f"/work/{CONFIG['paths']['sequences']}/binders.fasta",
             "--out_dir", f"/work/{CONFIG['paths']['boltz2_monomer']}"],
    mounts=[docker.Mount(REPO, "/work")],
    workdir="/work",
)
docker.run(spec, dry_run=DRY_RUN)

records = []
for key in survivors:
    design_id, sequence_id = key
    design_pdb = backbone_dir / f"{design_id}.pdb"
    try:
        records.append(parsers.parse_boltz2_monomer(
            boltz_dir, design_id, sequence_id,
            design_pdb=design_pdb if design_pdb.exists() else None,
            binder_chain=design_chain,
        ))
    except parsers.ParseError as exc:
        print(f"  parse failed for {sequence_id}: {exc}")

campaign.add_records(records)
survivors = campaign.gate("gate3b", CONFIG["gates"]["step3b_boltz2"],
                          "boltz2", enforce=not SMOKE_TEST)
print(f"\ngate 3B: {len(survivors)} passed")

In [ ]:
# Checkpoint 3: design pose (cyan) vs predicted monomer (magenta).
frame = campaign.frame()
if not frame.empty and "esmfold_pdb" in frame.columns:
    row = frame.dropna(subset=["esmfold_pdb"]).head(1)
    if not row.empty:
        r = row.iloc[0]
        design_pdb = backbone_dir / f"{r.design_id}.pdb"
        if design_pdb.exists():
            viz.write_pml_superposition(design_pdb, r.esmfold_pdb,
                                        REPO / "results/pml/03_monomer.pml",
                                        binder_chain=design_chain)
            display(viz.show_monomer_superposition(
                design_pdb, r.esmfold_pdb, scrmsd=r.get("esmfold_scrmsd")))

---

## **6. Complex and interface filtering (Step 4)**

Both engines run so their outputs can be compared.

`pae_interaction` is the primary discriminator, with ipTM secondary — it saturates less on de novo binders. It is computed as the mean of **both** off-diagonal inter-chain PAE blocks; the full-matrix mean would be dominated by the confident intra-chain blocks and would make a bad interface look good.

Note the one external dependency: ColabFold queries the **public MMseqs2 API**, so target sequences leave the host. Boltz-2 in single-sequence mode is the fully local alternative.

In [ ]:
# Build Step 4 inputs. The two engines need DIFFERENT formats -- ColabFold
# joins chains with ':' in one record, Boltz-2 wants one header per chain in
# its own file -- so a single shared complexes.fasta would fail for at least
# one of them, and silently: ColabFold fed a Boltz file sees one concatenated
# monomer and predicts the wrong thing confidently.
from pipeline import inputs

target_seq = inputs.target_sequence(target_pdb, chain_id)
print(f"target chain {chain_id}: {len(target_seq)} residues")

step4_samples = [s for s in all_samples if (s.design_id, s.sequence_id) in set(survivors)]
if not step4_samples:
    print("no Step 3 survivors to carry into Step 4 (expected in DRY_RUN)")
else:
    cf_fasta = inputs.write_colabfold_complex_fasta(
        step4_samples, target_seq, seq_dir / "complexes_colabfold.fasta",
        binder_chain_index=0,
    )
    boltz_files = inputs.write_boltz_complex_fastas(
        step4_samples, target_seq, seq_dir / "complexes_boltz",
        binder_chain_index=0,
    )
    print(f"wrote {cf_fasta.name} and {len(boltz_files)} Boltz input file(s)")

    # Controls. Without them a zero-lead outcome cannot be distinguished from a
    # broken pae_interaction.
    if CONFIG["controls"]["enabled"]:
        ctrl = inputs.write_controls(
            step4_samples[0].chain(0), target_seq, seq_dir / "controls",
        )
        print(f"controls: {sorted(ctrl)}")

In [ ]:
# NOTE on chain order: ColabFold receives binder-first, so ITS output has the
# binder as chain A and target as B -- the reverse of the RFdiffusion design
# PDB. Parsing below passes the chain ids explicitly for that reason.
complex_dir = REPO / CONFIG["paths"]["complexes"]
CF_BINDER, CF_TARGET = "A", "B"

for engine in CONFIG["step4"]["engines"]:
    image = CONFIG["containers"][engine]["image"]
    if engine == "boltz2":
        command = [f"/work/{CONFIG['paths']['sequences']}/complexes_boltz",
                   "--out_dir", f"/work/{CONFIG['paths']['complexes']}/boltz2"]
        if CONFIG["step4"]["boltz2_msa_mode"] != "single_sequence":
            # Sends target sequences to the public MMseqs2 server.
            command.append("--use_msa_server")
    else:
        command = [f"/work/{CONFIG['paths']['sequences']}/complexes_colabfold.fasta",
                   f"/work/{CONFIG['paths']['complexes']}/colabfold"]
    docker.run(
        docker.ContainerRun(image=image, command=command,
                            mounts=[docker.Mount(REPO, "/work")], workdir="/work"),
        dry_run=DRY_RUN,
    )

In [ ]:
# Parse both engines side by side. Engine-prefixed columns keep them from
# colliding, which is what makes the comparison possible.
for engine, parse in (("boltz2", parsers.parse_boltz2_complex),
                      ("colabfold", parsers.parse_colabfold_complex)):
    # Boltz input was written binder=A/target=B too, so both engines share
    # this layout -- which is NOT the design PDB's layout.
    records = []
    for design_id, sequence_id in survivors:
        design_pdb = backbone_dir / f"{design_id}.pdb"
        try:
            records.append(parse(
                complex_dir / engine, design_id, sequence_id,
                design_pdb=design_pdb if design_pdb.exists() else None,
                binder_chain=CF_BINDER, target_chain=CF_TARGET,
            ))
        except parsers.ParseError as exc:
            print(f"  {engine} parse failed for {sequence_id}: {exc}")
    campaign.add_records(records)

leads = campaign.gate("step4", CONFIG["gates"]["step4_complex"],
                      "boltz2", enforce=not SMOKE_TEST)
print(f"\nstep 4: {len(leads)} lead(s)")

In [ ]:
# Final summary, funnel, and the health check that distinguishes a bad campaign
# from a broken one.
summary = campaign.write(REPO / CONFIG["paths"]["summary"],
                         REPO / "results/funnel.csv")
print(f"wrote {summary}\n")

print("=== funnel ===")
print(campaign.funnel().to_string(index=False) if not campaign.funnel().empty else "empty")

print("\n=== health ===")
for key, value in campaign.health().items():
    print(f"  {key}: {value}")

print("\n=== top leads by pae_interaction ===")
ranked = campaign.rank("boltz2_pae_interaction")
if not ranked.empty:
    cols = [c for c in ("design_id", "sequence_id", "boltz2_iptm",
                        "boltz2_pae_interaction", "colabfold_iptm",
                        "colabfold_pae_interaction", "esmfold_plddt")
            if c in ranked.columns]
    print(ranked[cols].head(10).to_string(index=False))

In [ ]:
# Expected yield at full scale, before committing GPU hours.
print(report.funnel_projection(
    CONFIG["rfdiffusion"]["num_designs"]["full"],
    CONFIG["proteinmpnn"]["seqs_per_backbone"]["full"],
    [("gate3a", 0.17), ("gate3b", 0.07), ("step4", 0.5)],
).to_string(index=False))

In [ ]:
# Checkpoint 4: interface coloured by pLDDT, with polar contacts.
frame = campaign.frame()
pdb_col = next((c for c in ("boltz2_pdb", "colabfold_pdb") if c in frame.columns), None)
if pdb_col and not frame.dropna(subset=[pdb_col]).empty:
    best = campaign.rank("boltz2_pae_interaction").dropna(subset=[pdb_col]).iloc[0]
    viz.write_pml_interface(best[pdb_col], REPO / "results/pml/04_interface.pml",
                            binder_chain=design_chain, target_chain=fixed_chain)
    display(viz.show_complex_interface(best[pdb_col], binder_chain=design_chain,
                                       target_chain=fixed_chain))

---

## **Next**

Once this notebook runs clean end to end — every metric column populated,
controls behaving — the execution graph ports to the `Snakefile` in MANIFEST
Section 5. Two known issues to handle there, recorded in `MANIFEST_STATUS.md`:
`docker://` cannot resolve a local image (needs a `.sif` or a registry), and
`directory()` outputs defeat per-design parallelism.